In [ ]:
# ==============================================================================
# GLINKER — Medical Intake Pipeline
# main.ipynb  (orchestration only — all logic lives in glinker/ and pipeline.py)
#
# Run cells top-to-bottom on first use.
# On subsequent runs, skip Cell 5 (corpus already indexed) unless you want
# to rebuild the knowledge base from scratch.
# ==============================================================================


In [ ]:
# ── Cell 1: Pull code from GitHub ────────────────────────────────────────────
#
# The repo is cloned into /kaggle/working/curesense-project/ on first run.
# On later runs in the same session it does a git pull to get latest changes.
# REPO_DIR is added to sys.path so glinker/, pipeline.py, and api/ are
# importable immediately after the clone.
#
import sys, os, subprocess

REPO_URL  = 'https://github.com/moizaimran/curesense-project.git'
BRANCH    = 'hassan-branch'
REPO_DIR  = '/kaggle/working/curesense-project'

# If your repo is PRIVATE: add a Kaggle secret named 'GH_TOKEN'
# (Settings -> Secrets -> Add new secret, paste a GitHub Personal Access Token)
# and uncomment the two lines below:
# from kaggle_secrets import UserSecretsClient
# _token   = UserSecretsClient().get_secret('GH_TOKEN')
# REPO_URL = REPO_URL.replace('https://', f'https://{_token}@')

if not os.path.exists(REPO_DIR):
    print('Cloning repo ...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print('Repo already cloned — pulling latest ...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, REPO_DIR)
print('Ready:', REPO_DIR)


In [ ]:
# ── Cell 2: Installs ──────────────────────────────────────────────────────────
# requirements.txt covers the Flask service (pdfplumber, pypdfium2, openai,
# whisper, gliner, faiss, etc.).  The three lines below add the MedGemma
# FastAPI service dependencies that are not in requirements.txt.
!pip install -r {REPO_DIR}/requirements.txt -q
!pip install torchvision --upgrade --quiet
!pip install -q fastapi uvicorn pydicom

In [ ]:
# ── Cell 3: Secrets + OpenAI client ──────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok
from openai import OpenAI
import glinker.config as cfg

_secrets = UserSecretsClient()
ngrok.set_auth_token(_secrets.get_secret('Ngrok Key'))
cfg.openai_client = OpenAI(api_key=_secrets.get_secret('OpenAI Key'))
print('OpenAI client ready')


In [ ]:
# ── Cell 4: Load heavy models (Whisper + GLiNER) ──────────────────────────────
import torch
import whisper
from gliner import GLiNER
import glinker.config as cfg

print('Loading Whisper Large ...')
cfg.whisper_model = whisper.load_model('large', device='cuda')
print('Whisper ready')

print('Loading GLiNER BioMed ...')
cfg.gliner_model = GLiNER.from_pretrained('Ihor/gliner-biomed-bi-large-v1.0').to('cuda')
print('GLiNER ready on:', next(cfg.gliner_model.parameters()).device)


In [ ]:
# ── Cell 5: Load or build the RAG index ──────────────────────────────────────
#
# Priority order:
#   1. /kaggle/working/rag_index/                                    — already built this session
#   2. /kaggle/input/datasets/hassanraheem/uresense-rag-index/       — pre-built dataset
#   3. Build from scratch                                             — first time only
#
import os, shutil, glob
from glinker.rag.ingestion import build_index
import glinker.config as cfg

WORKING_INDEX = cfg.RAG_INDEX_DIR
DATASET_MOUNT = '/kaggle/input/datasets/hassanraheem/uresense-rag-index'

if os.path.exists(f"{WORKING_INDEX}/index.faiss"):
    print("Index already in working dir — skipping.")

else:
    # Search for index.faiss at any depth inside the dataset mount
    matches = glob.glob(f"{DATASET_MOUNT}/**/index.faiss", recursive=True)
    if not matches:
        matches = glob.glob(f"{DATASET_MOUNT}/index.faiss")

    if matches:
        source_dir = os.path.dirname(matches[0])
        print(f"Pre-built dataset found at: {source_dir} — copying to working dir ...")
        os.makedirs(WORKING_INDEX, exist_ok=True)
        for fname in ("index.faiss", "chunks.json"):
            shutil.copy(f"{source_dir}/{fname}", f"{WORKING_INDEX}/{fname}")
        print("Copied. Cell 6 will load it.")

    else:
        print(f"index.faiss not found under {DATASET_MOUNT}")
        print(f"Contents: {os.listdir(DATASET_MOUNT) if os.path.exists(DATASET_MOUNT) else 'dataset not mounted'}")
        print("Building from scratch (20-40 min) ...")
        build_index(textbooks_limit=5000, guidelines_limit=2000)
        print("Index built and saved to", WORKING_INDEX)


In [ ]:
# ── Cell 6: Load RAG index into memory ───────────────────────────────────────
from glinker.rag.retrieval import load_index
load_index()


In [ ]:
# ── Cell 7: Load disease ranking datasets (optional) ─────────────────────────
# Requires the 9 Kaggle symptom-disease datasets attached via Add Data.
# The pipeline degrades gracefully (no ranking) if none are attached.
from glinker.disease.ranker import load_datasets
load_datasets()


In [ ]:
# ── Cell 8: Launch Flask API on port 5001 (GPU 0) ────────────────────────────
#
# Whisper + GLiNER were loaded onto GPU 0 in Cell 4.  Flask runs here and
# inherits that GPU assignment — it stays on GPU 0.
#
# MedGemma FastAPI is launched in Cell 9 and pinned to GPU 1, so the two
# services never compete for memory.
#
# AI_SERVICE_URL (Flask)       → interview pipeline + PDF analysis
# MEDGEMMA_SERVICE_URL (Cell 9) → X-ray / CT / MRI via MedGemma 4B
#
from api.app import launch

API_URL = launch(port=5001)
print(f"Flask live — AI_SERVICE_URL={API_URL}")

In [ ]:
# ── Cell 9: Launch MedGemma FastAPI on port 5002 (GPU 1) ─────────────────────
#
# Dual-GPU split:
#   GPU 0 — Flask service  (Whisper large + GLiNER, loaded in Cell 4)
#   GPU 1 — MedGemma 4B   (pinned below via device_map={"": 1})
#
# If Kaggle provisions only 1 T4 this session, _load_model() detects it and
# falls back to device_map="auto" automatically — notebook won't hard-crash.
#
# After this cell runs you'll see per-GPU VRAM before/after the model loads.
# Copy both printed URLs into Backend/.env, then restart the Express server.
#
import threading
import uvicorn
from pyngrok import ngrok
from api.medgemma_app import app as medgemma_app, _load_model

# Start MedGemma FastAPI server in a background thread
_mg_thread = threading.Thread(
    target=lambda: uvicorn.run(medgemma_app, host="0.0.0.0", port=5002, log_level="warning"),
    daemon=True,
)
_mg_thread.start()
print("[MedGemma] FastAPI server started on port 5002\n")

# Pre-warm: load the model now so VRAM logs appear here in notebook output
# rather than silently on the first HTTP request.
_load_model()

# Expose MedGemma on its own ngrok tunnel
MEDGEMMA_URL = ngrok.connect(5002).public_url

print("\n" + "=" * 64)
print(f"  AI_SERVICE_URL       = {API_URL}")
print(f"  MEDGEMMA_SERVICE_URL = {MEDGEMMA_URL}")
print("=" * 64)
print("\nPaste both values into Backend/.env and restart Express.")